# Обзорный EDA по трём итоговым таблицам

Базовое описание данных для главы 2 (раздел «Финансовый аудит»): сколько записей, какой период покрыт, есть ли пропуски и нулевые/некорректные значения в ключевых полях трёх нормализованных таблиц — продажи по контрагентам, оплаты по контрагентам, банковские операции.

In [1]:
from pathlib import Path
import pandas as pd

BASE = Path("..").resolve()
DATA = BASE / "Данные" / "Итоговые таблицы"

sales = pd.read_parquet(DATA / "sales_by_counterparties.parquet")
payments = pd.read_parquet(DATA / "sales_by_payment_counterparties.parquet")
bank = pd.read_parquet(DATA / "bank_statement_transactions.parquet")

tables = {
    "sales_by_counterparties": (sales, "sale_date"),
    "sales_by_payment_counterparties": (payments, "payment_date"),
    "bank_statement_transactions": (bank, "operation_date"),
}

## Сводная таблица: объём, период, пропуски, нули

In [2]:
summary_rows = []
for name, (df, date_col) in tables.items():
    n_rows = len(df)
    n_cols = df.shape[1]
    date_min, date_max = df[date_col].min(), df[date_col].max()
    amount_col = "amount"
    n_amount_missing = df[amount_col].isna().sum()
    n_amount_zero = (df[amount_col] == 0).sum()
    n_amount_negative = (df[amount_col] < 0).sum()
    n_dup_rows = df.duplicated().sum()
    summary_rows.append({
        "таблица": name,
        "строк": n_rows,
        "колонок": n_cols,
        "период_с": date_min.date(),
        "период_по": date_max.date(),
        "amount_пропуски": n_amount_missing,
        "amount_нули": n_amount_zero,
        "amount_отрицательные": n_amount_negative,
        "дублирующиеся_строки": n_dup_rows,
    })

summary = pd.DataFrame(summary_rows)
display(summary)

,таблица,строк,колонок,период_с,период_по,amount_пропуски,amount_нули,amount_отрицательные,дублирующиеся_строки
0,sales_by_counterparties,3051,26,2025-05-06,2026-06-22,0,0,908,0
1,sales_by_payment_counterparties,3778,21,2025-05-05,2026-06-17,0,0,0,0
2,bank_statement_transactions,4622,20,2025-05-01,2026-06-19,0,0,831,0


## Пропуски по колонкам — построчно для каждой таблицы

In [3]:
for name, (df, _) in tables.items():
    print("====", name, "====")
    missing = df.isna().sum()
    missing = missing[missing > 0].sort_values(ascending=False)
    missing_pct = (missing / len(df) * 100).round(2)
    report = pd.DataFrame({"пропусков": missing, "доля_%": missing_pct})
    if report.empty:
        print("пропусков по колонкам нет\n")
    else:
        display(report)

==== sales_by_counterparties ====


,пропусков,доля_%
source_amount_column,3051,100.00
city_or_area,151,4.95
source_quantity_column,129,4.23
contract_date,2,0.07


==== sales_by_payment_counterparties ====


,пропусков,доля_%
city_or_area,287,7.6
brand,68,1.8
store_location_raw,68,1.8


==== bank_statement_transactions ====


,пропусков,доля_%
debtor_code,4622,100.00
counterparty_kpp,184,3.98


## Категориальный разрез: типы документов / направления операций

In [4]:
print("Типы документов продаж (sales_doc_type):")
display(sales["sales_doc_type"].value_counts())

print("\nНаправления банковских операций (direction):")
display(bank["direction"].value_counts())

print("\nЧисло уникальных контрагентов по таблицам:")
display(pd.Series({
    "sales_by_counterparties": sales["counterparty_raw"].nunique(),
    "sales_by_payment_counterparties": payments["counterparty_raw"].nunique(),
    "bank_statement_transactions": bank["counterparty_name"].nunique(),
}, name="уникальных_контрагентов"))

Типы документов продаж (sales_doc_type):


sales_doc_type
Реализация                  2143
Корректировка реализации     908
Name: count, dtype: Int64


Направления банковских операций (direction):


direction
credit    3791
debit      831
Name: count, dtype: Int64


Число уникальных контрагентов по таблицам:


sales_by_counterparties            191
sales_by_payment_counterparties    197
bank_statement_transactions        268
Name: уникальных_контрагентов, dtype: int64

## Контрольная сверка банковских оборотов

Дополнительная проверка: сумма дебета/кредита по строкам должна быть положительной и согласованной (хвостовая проверка после загрузочного пайплайна, см. главу 2).

In [5]:
print("Сумма debit:", bank["debit"].sum())
print("Сумма credit:", bank["credit"].sum())
print("Файлов-источников (sales):", sales["source_file"].nunique())
print("Файлов-источников (payments):", payments["source_file"].nunique())
print("Файлов-источников (bank):", bank["source_file"].nunique())

Сумма debit: 46498623.96
Сумма credit: 46388479.449999996
Файлов-источников (sales): 5
Файлов-источников (payments): 6
Файлов-источников (bank): 2
